# 체크포인트: MemorySaver와 SQLite — 정답본

원본: `src/solution/Chapter02/03-langgraph-advanced/02_persistence.py`  
슬라이드: `docs/raw/Chapter02/day01/3-4-langgraph-advanced.pdf`

## 학습 목표

- 기억 저장 수명·thread_id·Time-Travel
- 원본 코드를 CodeMate와 함께 읽고 실행 결과를 점검한다.


## 코드 흐름

그래프 → 메모리 대화 → SQLite 대화 → 이력 조회 → 과거 시점 재실행

**읽을 때 확인할 점:** 이전 회차 checkpoints.db는 분리했다. 원본 스크립트를 실행하면 새 DB가 생성될 수 있다. setup 셀은 임시 DB 경로를 써 기존 데이터를 건드리지 않는다. 커널 종료 후 임시 DB는 삭제된다.


## 실행 전 확인

LLM 호출과 SQLite 파일 쓰기가 일어난다. 실행 전에 setup 셀을 먼저 실행한다.

아래 원본 코드 셀은 실행을 직접 선택해야 하는 `manual-run` 셀입니다. 실행 준비 셀은 프로젝트 루트를 찾고 원본 `.py`의 경로를 `__file__`에 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "Chapter02").is_dir()),
    None,
)
if _repo_root is None:
    raise FileNotFoundError("저장소 안에서 이 노트북을 열어 주세요.")
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root / "src"))
__file__ = str(_repo_root / "src/solution/Chapter02/03-langgraph-advanced/02_persistence.py")
import tempfile
_notebook_checkpoint_dir = tempfile.TemporaryDirectory(prefix="langgraph_notebook_")
__file__ = str(Path(_notebook_checkpoint_dir.name) / "02_persistence.py")


## 원본 코드

다음 셀에는 원본 `.py` 전체가 그대로 들어 있습니다.


In [ ]:
import os
import sqlite3
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()


class State(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot(state: State) -> dict:
    system = SystemMessage(content='너는 친절한 AI 어시스턴트야.')
    return {'messages': [llm.invoke([system] + state['messages'])]}


builder = StateGraph(State)
builder.add_node('chatbot', chatbot)
builder.add_edge(START, 'chatbot')
builder.add_edge('chatbot', END)


# ── MemorySaver vs SqliteSaver ────────────────────────────────────────────────
#
# MemorySaver  : 프로세스 메모리에 저장 → 프로그램 재시작 시 기록 사라짐 (개발/데모용)
# SqliteSaver  : SQLite 파일에 저장   → 재시작해도 기록 유지 (로컬 프로토타입용)
# PostgresSaver: PostgreSQL에 저장    → 실제 서비스용 (langgraph-checkpoint-postgres 패키지)

print('=== 1. MemorySaver (휘발성) ===')
mem_app = builder.compile(checkpointer=MemorySaver())
# mem_app = builder.compile()
config = {'configurable': {'thread_id': 'demo'}}

mem_app.invoke({'messages': [HumanMessage(content='내 이름은 홍길동이야.')]}, config=config)
result = mem_app.invoke({'messages': [HumanMessage(content='내 이름이 뭐야?')]}, config=config)
print(result['messages'][-1].content)


print('\n=== 2. SqliteSaver (영속성) ===')
DB_PATH = os.path.join(os.path.dirname(__file__), 'checkpoints.db')

with sqlite3.connect(DB_PATH, check_same_thread=False) as conn:
    sqlite_app = builder.compile(checkpointer=SqliteSaver(conn))
    config_sql = {'configurable': {'thread_id': 'persistent_demo'}}

    sqlite_app.invoke({'messages': [HumanMessage(content='내 취미는 독서야.')]}, config=config_sql)
    result_sql = sqlite_app.invoke({'messages': [HumanMessage(content='내 취미가 뭐야?')]}, config=config_sql)
    print(result_sql['messages'][-1].content)

    # 저장된 체크포인트 목록 확인
    checkpoints = list(sqlite_app.get_state_history(config_sql))
    print(f'\n저장된 체크포인트 수: {len(checkpoints)}')
    for cp in checkpoints[:3]:
        print(f'  - {cp.config["configurable"]["checkpoint_id"][:8]}... | 메시지 수: {len(cp.values["messages"])}')

    # Time-Travel: 특정 체크포인트로 되돌아가 다른 흐름으로 재실행
    if len(checkpoints) >= 2:
        past_config = checkpoints[-1].config   # 가장 오래된 체크포인트
        print(f'\n=== Time-Travel: 과거 시점에서 다른 질문으로 재실행 ===')
        result_tt = sqlite_app.invoke(
            {'messages': [HumanMessage(content='독서 말고 다른 취미도 있어?')]},
            config=past_config,
        )
        print(result_tt['messages'][-1].content)


## CodeMate와 함께 검토

> 같은 thread_id와 새 thread_id의 기록 차이를 예측하고 get_state_history와 past_config 선택을 검토해 줘. 원본은 바꾸지 말고 검토 내용을 먼저 말해 줘.

제안된 변경은 어떤 문제를 해결하는지, 어떤 결과로 확인할지 설명해 보세요.

## 확인 질문

1. MemorySaver 기록은 언제 사라지는가?
2. checkpoints[0]은 어느 시점인가?
